# Advent of Code 2025

## Day 10

https://adventofcode.com/2025/day/10

### Part 1

In [1]:
from collections import defaultdict, deque
import time
from plotly import graph_objects as go
import numpy as np
from typing import List

day = 10

test_result = 7
test_data = """\
[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}
[...#.] (0,2,3,4) (2,3) (0,4) (0,1,2) (1,2,3,4) {7,5,12,7,2}
[.###.#] (0,1,2,3,4) (0,3,4) (0,1,2,4,5) (1,2) {10,11,11,5,10,5}
"""

In [2]:
def solution1(data):
    lines = data.split("\n")[:-1]
    
    output = 0
    for line in lines:
        line = line.split(" ")
        target = int("0b" + line[0][1:-1][::-1].replace("#", "1").replace(".", "0"), base=0)
        buttons = [[int(_b) for _b in b[1:-1].split(",")] for b in line[1:-1]]
        buttons = [sum([1 << _b for _b in b]) for b in buttons]

        q = deque([(0, b, []) for b in buttons])    

        go_on = True
        #print(f"target: {target}")
        while q and go_on:
            curr, button, hist = q.popleft()
            #print(curr, button, hist)
            hist.append(button)
            curr = curr ^ button
            
            if curr == target:
                go_on = False
                output += len(hist)
                #print(f"found solution: {hist} with length {len(hist)}")        
                break
            else:
                for b in buttons:
                    if b == button:
                        continue
                    q.append((curr, b, hist.copy()))
    return output

    
sol1 = solution1(test_data)
print(sol1)
assert sol1 == test_result
print("test passed")

7
test passed


In [3]:
with open(f"Day{day:02d}.txt") as f:
    inp_data = f.read()
t0 = time.time()
sol1 = solution1(inp_data)
print(f"elapsed time {time.time()-t0:.2f} s")
print(sol1)
assert sol1 == 578
print("test passed")

elapsed time 73.85 s
578
test passed


### Part 2

In [4]:
test_result2 = 33

In [5]:
from z3 import Optimize, Int, Sum, sat

def find_min_pushes2(targets, buttons):
    # https://gitlab.com/0xdf/aoc2025/-/blob/main/day10/day10.py?ref_type=heads
    opt = Optimize()
    x = [Int(f"x{i}") for i in range(len(buttons))]
    for xi in x:
        opt.add(xi >= 0)

    for i, target in enumerate(targets):
        coef = [int(i in button) for button in buttons]
        opt.add(Sum(xi * c for c, xi in zip(coef, x)) == target)

    opt.minimize(Sum(x))

    if opt.check() == sat:
        m = opt.model()
        return sum(m[xi].as_long() for xi in x)

def solution2(data):
    lines = data.split("\n")[:-1]
    
    output = 0
    for line in lines:
        target_s, *buttons_s, power_s = line.split()
        buttons2 = [list(map(int, b.strip("()").split(","))) for b in buttons_s]
        target2 = list(map(int, power_s.strip("{}").split(",")))
        output += find_min_pushes2(target2, buttons2)
            

    return output
    
t0 = time.time()
sol2 = solution2(test_data)
print(f"elapsed time {time.time()-t0:.2f} s")
print(sol2)

assert sol2 == test_result2
print("test passed")

elapsed time 0.09 s
33
test passed


In [6]:
with open(f"Day{day:02d}.txt") as f:
    inp_data = f.read()

t0 = time.time()
sol2 = solution2(inp_data)
print(f"elapsed time {time.time()-t0:.2f} s")

print(sol2)

assert sol2 == 20709
print("test passed")

elapsed time 1.53 s
20709
test passed
